# Career Orientation Classifier

This notebook trains a classifier that maps student profiles to one of 6 career branches:
1. Software Engineering
2. Data Science
3. AI/ML
4. Web Development
5. Business Analytics
6. UX/UI Design

**Pipeline:** Gemini 2.5 Flash generates synthetic profiles → sentence-transformers encodes text → frozen encoder + trainable classification head.

## Cell 1 — Imports & Setup

Install dependencies and import all libraries. Load the Gemini API key from a `.env` file located one level above this notebook.

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'google-generativeai',
    'sentence-transformers',
    'torch',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'python-dotenv'
], check=True)
print('All packages installed.')

In [ ]:
import os
import json
import time
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import google.generativeai as genai
from dotenv import load_dotenv

# ── Paths (all relative to notebook location) ────────────────────────────────
NOTEBOOK_DIR = Path('.')
DATA_DIR     = NOTEBOOK_DIR / '..' / 'data'
MODELS_DIR   = NOTEBOOK_DIR / '..' / 'models'
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

RAW_DATA_PATH  = DATA_DIR / 'raw_profiles.json'
LABEL_MAP_PATH = DATA_DIR / 'label_map.json'
BEST_MODEL_PATH  = MODELS_DIR / 'best_model.pt'
FINAL_MODEL_PATH = MODELS_DIR / 'career_model.pt'

# ── Device ───────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Gemini API key ────────────────────────────────────────────────────────────
# Load from .env file located at career_orientation/.env (one dir above notebooks/)
env_path = NOTEBOOK_DIR / '..' / '.env'
load_dotenv(dotenv_path=env_path)
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
if not GEMINI_API_KEY:
    raise EnvironmentError(
        'GEMINI_API_KEY not found. '
        'Create career_orientation/.env with: GEMINI_API_KEY=your_key_here'
    )
genai.configure(api_key=GEMINI_API_KEY)
print('Gemini API configured.')

# ── Constants ─────────────────────────────────────────────────────────────────
CATEGORIES = [
    'Software Engineering',
    'Data Science',
    'AI/ML',
    'Web Development',
    'Business Analytics',
    'UX/UI Design',
]
PROFILES_PER_CLASS = 200
BATCH_SIZE_FULL    = 15
SEED               = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Setup complete.')

## Cell 2 — Synthetic Data Generation

Use Gemini 2.5 Flash to generate 200 student profiles per career class (1 200 total).
Each API call produces 15 profiles (except the last call per class which produces 5).
Generation is skipped if `data/raw_profiles.json` already exists.

In [ ]:
gemini_model = genai.GenerativeModel('gemini-2.5-flash')

def strip_markdown(text: str) -> str:
    """Remove ```json ... ``` fences that Gemini sometimes adds."""
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    return text.strip()


def generate_batch(category: str, batch_num: int, count: int = 15) -> list[dict]:
    """
    Call Gemini to generate `count` synthetic student profiles for `category`.
    `batch_num` is embedded in the prompt so the model produces diverse outputs.
    Returns a list of profile dicts.
    """
    prompt = f"""
You are generating a DIVERSE synthetic dataset for a career orientation classifier.

Task: Generate exactly {count} student profiles oriented toward the career: **{category}**.
This is batch {batch_num} — profiles MUST be noticeably different from previous batches.

Diversity requirements:
- Vary skill level: beginner / intermediate / advanced
- Vary background: self-taught / university / bootcamp / vocational
- Vary age: between 18 and 28
- Vary geographic region: Europe, Africa, Southeast Asia, Latin America, North America, Middle East
- Vary academic strengths and weaknesses realistically

Return a JSON array of {count} objects. Each object must have EXACTLY these fields:
{{
  "skills": ["skill1", ...],          // list of 3-8 technical or soft skills
  "interests": ["interest1", ...],    // list of 2-5 personal/professional interests
  "academic_performance": {{          // scores 0-20
    "math": 0-20,
    "sciences": 0-20,
    "languages": 0-20,
    "arts": 0-20
  }},
  "projects": ["desc1", ...],         // list of 1-3 project descriptions
  "goals": {{
    "salary_expectation": "<value>",  // e.g. "35000", "60000", "90000"
    "remote_preference": "<value>",   // "remote", "hybrid", or "on-site"
    "target_field": "<value>"         // e.g. "fintech", "healthcare", "gaming"
  }},
  "label": "{category}"
}}

Return ONLY the raw JSON array. No markdown fences, no explanation.
"""
    response = gemini_model.generate_content(prompt)
    raw = strip_markdown(response.text)
    profiles = json.loads(raw)
    # Ensure label is set correctly regardless of what Gemini returned
    for p in profiles:
        p['label'] = category
    return profiles


if RAW_DATA_PATH.exists():
    print(f'raw_profiles.json already exists — skipping generation. Delete it to re-generate.')
else:
    all_profiles = []
    for category in CATEGORIES:
        # 13 full batches of 15 = 195 profiles, then 1 batch of 5 = 200 total
        batches = [(i + 1, BATCH_SIZE_FULL) for i in range(13)] + [(14, 5)]
        cat_profiles = []
        for batch_num, count in batches:
            try:
                batch = generate_batch(category, batch_num, count)
                cat_profiles.extend(batch)
                print(f'  [{category}] batch {batch_num:02d}/14 — got {len(batch)} profiles '
                      f'(total so far: {len(cat_profiles)})')
            except Exception as e:
                print(f'  [{category}] batch {batch_num} FAILED: {e}')
            time.sleep(1)
        all_profiles.extend(cat_profiles)
        print(f'✓ {category}: {len(cat_profiles)} profiles generated\n')

    RAW_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(RAW_DATA_PATH, 'w', encoding='utf-8') as f:
        json.dump(all_profiles, f, ensure_ascii=False, indent=2)
    print(f'Saved {len(all_profiles)} profiles to {RAW_DATA_PATH}')

## Cell 3 — Data Validation & EDA

Load the generated profiles, verify class balance, check for near-duplicates, and visualize the distribution.

In [ ]:
with open(RAW_DATA_PATH, 'r', encoding='utf-8') as f:
    raw_profiles = json.load(f)

df_raw = pd.DataFrame(raw_profiles)
print(f'Total profiles loaded: {len(df_raw)}')
print()

# ── Per-class distribution ────────────────────────────────────────────────────
class_counts = df_raw['label'].value_counts()
print('Per-class distribution:')
print(class_counts.to_string())
print()

# ── Near-duplicate check (same top-3 skills) ─────────────────────────────────
def skill_key(profile):
    skills = profile.get('skills', [])
    top3 = sorted([s.lower().strip() for s in skills])[:3]
    return '|'.join(top3)

df_raw['_skill_key'] = df_raw.apply(skill_key, axis=1)
dup_mask = df_raw.duplicated(subset=['_skill_key'], keep=False)
n_dups = dup_mask.sum()
print(f'Profiles sharing the same top-3 skills as another profile: {n_dups}')
if n_dups > 0:
    print('  (These are flagged as potential near-duplicates — review if needed)')
df_raw = df_raw.drop(columns=['_skill_key'])
print()

# ── Plot 1: Class distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    x=class_counts.index, y=class_counts.values,
    palette='viridis', ax=axes[0]
)
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_xlabel('Career Branch')
axes[0].set_ylabel('Number of Profiles')
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(axes[0].patches, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 str(val), ha='center', va='bottom', fontsize=9)

# ── Plot 2: Academic score distributions ─────────────────────────────────────
acad_rows = []
for _, row in df_raw.iterrows():
    acad = row.get('academic_performance', {})
    if isinstance(acad, dict):
        for subject, score in acad.items():
            acad_rows.append({'subject': subject, 'score': score})
df_acad = pd.DataFrame(acad_rows)

sns.boxplot(data=df_acad, x='subject', y='score', palette='Set2', ax=axes[1])
axes[1].set_title('Academic Score Distribution by Subject', fontsize=13)
axes[1].set_ylabel('Score (0–20)')
axes[1].set_xlabel('Subject')

plt.tight_layout()
plt.show()

# ── Sample profile per class ──────────────────────────────────────────────────
print('\n── Sample profile per class ──')
for cat in CATEGORIES:
    sample = df_raw[df_raw['label'] == cat].iloc[0].to_dict()
    sample.pop('_skill_key', None)
    print(f'\n[{cat}]')
    print(json.dumps(sample, indent=2, ensure_ascii=False))

## Cell 4 — Preprocessing

Convert structured profile dicts to flat text strings, encode labels, and split into train / val / test sets (70 / 15 / 15).

In [ ]:
def profile_to_text(profile: dict) -> str:
    """Serialize a profile dict to a fixed-format text string."""
    skills    = ', '.join(profile.get('skills', []))
    interests = ', '.join(profile.get('interests', []))
    projects  = '. '.join(profile.get('projects', []))
    goals     = profile.get('goals', {})
    acad      = profile.get('academic_performance', {})

    target_field       = goals.get('target_field', 'N/A')
    salary_expectation = goals.get('salary_expectation', 'N/A')
    remote_preference  = goals.get('remote_preference', 'N/A')

    math      = acad.get('math', 0)
    sciences  = acad.get('sciences', 0)
    languages = acad.get('languages', 0)
    arts      = acad.get('arts', 0)

    return (
        f"Skills: {skills}. "
        f"Interests: {interests}. "
        f"Projects: {projects}. "
        f"Goals: target {target_field}, salary {salary_expectation}, remote {remote_preference}. "
        f"Academic: Math {math}/20, Sciences {sciences}/20, "
        f"Languages {languages}/20, Arts {arts}/20."
    )


# ── Build DataFrame ───────────────────────────────────────────────────────────
with open(RAW_DATA_PATH, 'r', encoding='utf-8') as f:
    raw_profiles = json.load(f)

texts  = [profile_to_text(p) for p in raw_profiles]
labels = [p['label'] for p in raw_profiles]

# Label encoding: sorted alphabetically → 0-5
sorted_classes = sorted(set(labels))
label_to_id = {cls: idx for idx, cls in enumerate(sorted_classes)}
id_to_label = {idx: cls for cls, idx in label_to_id.items()}
label_ids   = [label_to_id[l] for l in labels]

print('Label mapping (alphabetical order):')
for cls, idx in label_to_id.items():
    print(f'  {idx}: {cls}')

# Save label map
with open(LABEL_MAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(label_to_id, f, indent=2)
print(f'\nLabel map saved to {LABEL_MAP_PATH}')

df = pd.DataFrame({'text': texts, 'label': labels, 'label_id': label_ids})

# ── Train / Val / Test split (70 / 15 / 15) stratified ───────────────────────
X, y = df['text'].tolist(), df['label_id'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print(f'\nSplit sizes:')
print(f'  Train : {len(X_train)} samples')
print(f'  Val   : {len(X_val)} samples')
print(f'  Test  : {len(X_test)} samples')

# Quick sanity check: show one converted text
print('\nSample converted text:')
print(X_train[0])

## Cell 5 — Model Definition

`CareerClassifier` wraps a **frozen** `all-MiniLM-L6-v2` encoder (384-dim output) with a trainable 3-layer classification head ending at 6 logits.

In [ ]:
class CareerClassifier(nn.Module):
    def __init__(self, num_classes: int = 6):
        super().__init__()
        # Frozen sentence encoder
        self.encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        for param in self.encoder.parameters():
            param.requires_grad = False

        # Trainable classification head
        self.classifier = nn.Sequential(
            nn.Linear(384, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, texts: list[str]) -> torch.Tensor:
        embeddings = self.encoder.encode(
            texts,
            convert_to_tensor=True,
            device=device,
            show_progress_bar=False,
        )
        return self.classifier(embeddings)


# Instantiate and inspect
model = CareerClassifier(num_classes=len(label_to_id)).to(device)

print('Model architecture:')
print(model.classifier)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Frozen parameters   : {total_params - trainable_params:,}')

## Cell 6 — Training

Train for 20 epochs with AdamW, tracking loss and validation accuracy each epoch. The best checkpoint (by val accuracy) is saved to `models/best_model.pt`.

In [ ]:
# ── Dataset & DataLoader ──────────────────────────────────────────────────────
class CareerDataset(Dataset):
    def __init__(self, texts: list[str], labels: list[int]):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


TRAIN_BATCH = 32

train_loader = DataLoader(
    CareerDataset(X_train, y_train),
    batch_size=TRAIN_BATCH, shuffle=True
)
val_loader = DataLoader(
    CareerDataset(X_val, y_val),
    batch_size=TRAIN_BATCH, shuffle=False
)

# ── Loss, optimizer ───────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-4, weight_decay=1e-4
)

# ── Training loop ─────────────────────────────────────────────────────────────
EPOCHS = 20
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for texts_batch, labels_batch in train_loader:
        labels_batch = torch.tensor(labels_batch, dtype=torch.long).to(device)
        optimizer.zero_grad()
        logits = model(list(texts_batch))
        loss   = criterion(logits, labels_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(texts_batch)
    train_loss = running_loss / len(X_train)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss_sum = 0.0
    val_preds, val_true = [], []
    with torch.no_grad():
        for texts_batch, labels_batch in val_loader:
            labels_tensor = torch.tensor(labels_batch, dtype=torch.long).to(device)
            logits = model(list(texts_batch))
            val_loss_sum += criterion(logits, labels_tensor).item() * len(texts_batch)
            preds = logits.argmax(dim=1).cpu().tolist()
            val_preds.extend(preds)
            val_true.extend(list(labels_batch))
    val_loss = val_loss_sum / len(X_val)
    val_acc  = accuracy_score(val_true, val_preds)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # ── Save best model ────────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        saved_marker = ' ✓ saved'
    else:
        saved_marker = ''

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'Train Loss: {train_loss:.4f} | '
        f'Val Loss: {val_loss:.4f} | '
        f'Val Acc: {val_acc:.4f}'
        f'{saved_marker}'
    )

print(f'\nBest validation accuracy: {best_val_acc:.4f}')

# ── Training curves ───────────────────────────────────────────────────────────
epochs_x = list(range(1, EPOCHS + 1))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_x, history['train_loss'], label='Train Loss', marker='o', markersize=3)
ax1.plot(epochs_x, history['val_loss'],   label='Val Loss',   marker='o', markersize=3)
ax1.set_title('Loss over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_x, history['val_acc'], label='Val Accuracy', color='green', marker='o', markersize=3)
ax2.axhline(y=0.85, linestyle='--', color='red', alpha=0.5, label='Target (85%)')
ax2.set_title('Validation Accuracy over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 7 — Evaluation

Load the best checkpoint and evaluate on the held-out test set. Target: ≥ 85% accuracy.

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
with open(LABEL_MAP_PATH, 'r') as f:
    label_to_id = json.load(f)
id_to_label = {v: k for k, v in label_to_id.items()}
num_classes  = len(label_to_id)

eval_model = CareerClassifier(num_classes=num_classes).to(device)
eval_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
eval_model.eval()
print(f'Loaded best model from {BEST_MODEL_PATH}')

# ── Run on test set ───────────────────────────────────────────────────────────
test_loader = DataLoader(
    CareerDataset(X_test, y_test),
    batch_size=32, shuffle=False
)

all_preds, all_true = [], []
with torch.no_grad():
    for texts_batch, labels_batch in test_loader:
        logits = eval_model(list(texts_batch))
        preds  = logits.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_true.extend(list(labels_batch))

test_accuracy = accuracy_score(all_true, all_preds)
print(f'\nTest Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.1f}%)')
if test_accuracy >= 0.85:
    print('Target of 85% reached!')
else:
    print(f'Below 85% target — consider more epochs or data augmentation.')

target_names = [id_to_label[i] for i in range(num_classes)]
print('\nClassification Report:')
print(classification_report(all_true, all_preds, target_names=target_names))

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=target_names, yticklabels=target_names
)
plt.title('Confusion Matrix — Test Set', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Cell 8 — Inference Function

`predict_career(profile_dict)` returns the top-3 predicted career branches with confidence scores. Tested on two hardcoded profiles.

In [ ]:
# ── Load artefacts ────────────────────────────────────────────────────────────
with open(LABEL_MAP_PATH, 'r') as f:
    label_to_id = json.load(f)
id_to_label = {v: k for k, v in label_to_id.items()}

infer_model = CareerClassifier(num_classes=len(label_to_id)).to(device)
infer_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
infer_model.eval()


def predict_career(profile_dict: dict) -> dict:
    """
    Given a student profile dict, return a dict with:
      - 'top_prediction': the most likely career label
      - 'top3': list of (label, confidence_pct) for the top 3 predictions
    """
    text = profile_to_text(profile_dict)
    with torch.no_grad():
        logits = infer_model([text])
        probs  = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

    top3_indices = probs.argsort()[::-1][:3]
    top3 = [
        (id_to_label[int(idx)], round(float(probs[idx]) * 100, 2))
        for idx in top3_indices
    ]
    return {
        'top_prediction': top3[0][0],
        'top3': top3,
    }


# ── Test profile 1: obvious AI/ML candidate ───────────────────────────────────
profile_aiml = {
    'skills': ['Python', 'PyTorch', 'TensorFlow', 'scikit-learn', 'Linear Algebra'],
    'interests': ['deep learning', 'computer vision', 'research'],
    'academic_performance': {'math': 19, 'sciences': 18, 'languages': 12, 'arts': 8},
    'projects': [
        'Implemented ResNet-50 from scratch for image classification',
        'Fine-tuned BERT for sentiment analysis on Twitter data',
    ],
    'goals': {
        'salary_expectation': '85000',
        'remote_preference': 'remote',
        'target_field': 'autonomous systems',
    },
    'label': 'AI/ML',
}

# ── Test profile 2: ambiguous (could be Data Science or Business Analytics) ───
profile_ambiguous = {
    'skills': ['SQL', 'Excel', 'Power BI', 'Python basics', 'statistics'],
    'interests': ['market trends', 'data storytelling', 'business strategy'],
    'academic_performance': {'math': 15, 'sciences': 13, 'languages': 16, 'arts': 11},
    'projects': [
        'Built a sales dashboard for a local SME using Power BI',
        'Performed customer segmentation analysis for an e-commerce dataset',
    ],
    'goals': {
        'salary_expectation': '48000',
        'remote_preference': 'hybrid',
        'target_field': 'retail analytics',
    },
    'label': 'ambiguous (Data Science / Business Analytics)',
}

# ── Print results ─────────────────────────────────────────────────────────────
for name, profile in [
    ('Obvious AI/ML profile', profile_aiml),
    ('Ambiguous profile (DS vs BA)', profile_ambiguous),
]:
    result = predict_career(profile)
    print(f'\n── {name} ──')
    print(f"  Top prediction : {result['top_prediction']}")
    print('  Top 3 rankings :')
    for rank, (label, pct) in enumerate(result['top3'], 1):
        bar = '█' * int(pct / 2)
        print(f'    {rank}. {label:<25} {pct:5.1f}%  {bar}')

## Cell 9 — Save & Export

Persist the final model state dict, re-save the label map, and print a compact summary.

In [ ]:
# ── Save final model ──────────────────────────────────────────────────────────
torch.save(infer_model.state_dict(), FINAL_MODEL_PATH)
print(f'Final model saved to {FINAL_MODEL_PATH}')

# ── Re-save label map (idempotent) ────────────────────────────────────────────
with open(LABEL_MAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(label_to_id, f, indent=2)
print(f'Label map saved to {LABEL_MAP_PATH}')

# ── Model size ────────────────────────────────────────────────────────────────
model_size_mb = FINAL_MODEL_PATH.stat().st_size / (1024 ** 2)

# ── Retrieve stored metrics (set in Cell 6 & 7 if run in the same kernel) ─────
try:
    train_acc_final = max(history['val_acc'])   # best val acc during training
except NameError:
    train_acc_final = float('nan')

try:
    _test_acc = test_accuracy
except NameError:
    # Re-evaluate if cells were not run in the same session
    all_preds2, all_true2 = [], []
    infer_model.eval()
    with open(RAW_DATA_PATH, 'r') as f:
        _raw = json.load(f)
    # Rebuild test split
    _texts  = [profile_to_text(p) for p in _raw]
    _labels = [label_to_id[p['label']] for p in _raw]
    _, _X_temp, _, _y_temp = train_test_split(_texts, _labels, test_size=0.30, stratify=_labels, random_state=SEED)
    _, _X_test, _, _y_test = train_test_split(_X_temp, _y_temp, test_size=0.50, stratify=_y_temp, random_state=SEED)
    _loader = DataLoader(CareerDataset(_X_test, _y_test), batch_size=32)
    with torch.no_grad():
        for tb, lb in _loader:
            logits = infer_model(list(tb))
            all_preds2.extend(logits.argmax(dim=1).cpu().tolist())
            all_true2.extend(list(lb))
    _test_acc = accuracy_score(all_true2, all_preds2)

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print('=' * 50)
print('        MODEL SUMMARY')
print('=' * 50)
print(f'  Encoder        : sentence-transformers/all-MiniLM-L6-v2 (frozen)')
print(f'  Classes        : {num_classes}')
print(f'  Best Val Acc   : {train_acc_final:.4f} ({train_acc_final*100:.1f}%)')
print(f'  Test Accuracy  : {_test_acc:.4f} ({_test_acc*100:.1f}%)')
print(f'  Model Size     : {model_size_mb:.2f} MB')
print(f'  Saved to       : {FINAL_MODEL_PATH.resolve()}')
print('=' * 50)